# MIE 402 — Pre-Lab 1: Sampling a Two-Tone Signal

**Fall 2026 — Dynamic Data Sampling and Frequency Analysis**

This pre-lab prepares you to recognize adequate sampling, aliasing, and FFT peaks before you collect sound data with the Moku:Go.

## Submission

- Run every code cell and keep all figures visible.
- Type short answers in the response cells.
- Export the completed notebook to HTML or PDF and submit it on Canvas **before your own laboratory section begins**.
- The pre-lab is normally posted on Canvas on the Monday before the laboratory. Late pre-labs are not accepted.

You may discuss general approaches, but your submitted notebook and explanations must be your own work.


## New problem

Consider the two-tone voltage signal

$$x(t)=1.20\sin(2\pi(18)t+25^\circ)+0.45\cos(2\pi(42)t-15^\circ)\quad\text{V}.$$

The 18 Hz component represents the desired signal and the 42 Hz component represents a second tone that could be present in a measurement.

1. Calculate the theoretical mean and RMS over a complete one-second record.
2. Sample the same signal for 1.00 s at **336, 126, 72, and 48 Hz**. Compare the samples with a dense reference waveform and calculate the sampled mean and RMS.
3. Compute a one-sided FFT magnitude spectrum for every sampled record.
4. Identify which records reproduce both frequencies correctly. When aliasing occurs, state the false frequency that appears.
5. Explain why a time-domain plot can look plausible even when its frequency spectrum is wrong.

These values and the two-tone signal differ from the Spring 2026 pre-lab, while practicing the same concepts needed in Lab 1.


## 1. Define the signal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11})

# Signal definition
A1, f1, phi1_deg = 1.20, 18.0, 25.0
A2, f2, phi2_deg = 0.45, 42.0, -15.0
duration = 1.00
sample_rates = [336.0, 126.0, 72.0, 48.0]

def signal(t):
    phi1 = np.deg2rad(phi1_deg)
    phi2 = np.deg2rad(phi2_deg)
    return (A1*np.sin(2*np.pi*f1*t + phi1)
            + A2*np.cos(2*np.pi*f2*t + phi2))

def mean_and_rms(x):
    return np.mean(x), np.sqrt(np.mean(x**2))

def one_sided_spectrum(x, fs):
    n = len(x)
    window = np.hanning(n)
    coherent_gain = np.mean(window)
    magnitude = np.abs(np.fft.rfft(x*window))/(n*coherent_gain)
    if n > 2:
        magnitude[1:-1] *= 2
    frequency = np.fft.rfftfreq(n, d=1/fs)
    return frequency, magnitude


## 2. Theoretical reference

In [ ]:
# Dense reference waveform and theoretical values
t_ref = np.linspace(0, duration, 20001, endpoint=False)
x_ref = signal(t_ref)

theoretical_mean = 0.0
theoretical_rms = np.sqrt((A1**2 + A2**2)/2)

print(f"Theoretical mean = {theoretical_mean:.4f} V")
print(f"Theoretical RMS  = {theoretical_rms:.4f} V")

plt.plot(t_ref, x_ref)
plt.xlim(0, 0.20)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Dense reference waveform — first 0.20 s")
plt.grid(True)
plt.show()


## 3. Sampled time histories

In [ ]:
# Sample the signal at four rates and compare it with the reference
records = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, fs in zip(axes.flat, sample_rates):
    n = int(round(duration*fs))
    t = np.arange(n)/fs
    x = signal(t)
    records[fs] = (t, x)
    sample_mean, sample_rms = mean_and_rms(x)

    ax.plot(t_ref, x_ref, color="0.75", lw=1.2, label="Reference")
    ax.plot(t, x, "o-", ms=3, lw=0.9, label="Samples")
    ax.set_xlim(0, 0.20)
    ax.set_title(f"fs = {fs:g} Hz | mean={sample_mean:.4f} V | RMS={sample_rms:.4f} V")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Voltage (V)")
    ax.grid(True)

axes.flat[0].legend()
fig.suptitle("Time-domain comparison", fontsize=14)
fig.tight_layout()
plt.show()


## 4. Frequency-domain analysis

In [ ]:
# Compute and plot the one-sided spectra
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, fs in zip(axes.flat, sample_rates):
    _, x = records[fs]
    freq, mag = one_sided_spectrum(x, fs)
    ax.stem(freq, mag, basefmt=" ")
    ax.axvline(f1, color="tab:green", ls="--", lw=1, label="18 Hz expected")
    ax.axvline(f2, color="tab:red", ls="--", lw=1, label="42 Hz expected")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 1.35)
    ax.set_title(f"fs = {fs:g} Hz; Nyquist = {fs/2:g} Hz")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (V)")
    ax.grid(True)

axes.flat[0].legend(loc="upper right")
fig.suptitle("One-sided FFT magnitude spectra", fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
# List the strongest spectral peaks to support your interpretation
for fs in sample_rates:
    _, x = records[fs]
    freq, mag = one_sided_spectrum(x, fs)
    candidate = np.argsort(mag[1:])[-4:] + 1
    candidate = candidate[np.argsort(mag[candidate])[::-1]]
    strongest = [(round(float(freq[i]), 2), round(float(mag[i]), 3)) for i in candidate]
    print(f"fs={fs:6.1f} Hz, Nyquist={fs/2:5.1f} Hz, strongest bins={strongest}")


## Your responses

### 1. Mean and RMS

Show the theoretical calculation. Why can the RMS values of the two components be combined by adding their **squares**?

**Response:**  


### 2. Time-domain sampling

Compare the sampled mean and RMS values with theory. Which sampling rate gives the most convincing time-domain trace, and why?

**Response:**  


### 3. FFT and aliasing

For each sampling rate, list the two dominant frequencies. Which cases recover 18 Hz and 42 Hz correctly? For each aliased case, calculate the expected false frequency using

$$f_{alias}=|f-kf_s|,$$

where integer $k$ is chosen so that $f_{alias}$ lies between 0 and the Nyquist frequency.

**Response:**  


### 4. Engineering interpretation

In 3–5 sentences, explain why checking only a time history is insufficient when validating dynamic experimental data. Connect your explanation to the Moku:Go sound measurements in Lab 1.

**Response:**  
